# Hexene epoxidation — NEB coadsorption model

Input data retained. Rate/coverage plots use the active R/P descriptor grid; C/O/H are **not** present as variable descriptors in the current energy table.

The new plotting code converts CatMAP high-precision outputs to float to avoid Matplotlib dtype errors.


In [ ]:
from pathlib import Path
import os
from catmap import ReactionModel

project = Path.cwd()
if not (project / 'alkene_epoxidation.mkm').is_file():
    candidates = [project / 'lyq_alkene_epoxidation1' / '2-creating_microkinetic_model',
                  project / '2-creating_microkinetic_model']
    matches = [p for p in candidates if (p / 'alkene_epoxidation.mkm').is_file()]
    if not matches:
        raise FileNotFoundError('Open the new model directory. Current directory: ' + str(project))
    project = matches[0]
os.chdir(project)

model = ReactionModel(setup_file='alkene_epoxidation.mkm')
for variable in ['production_rate', 'rate', 'coverage']:
    if variable not in model.output_variables:
        model.output_variables.append(variable)

# CatMAP text-header serialization cannot eval certain numpy/mpmath repr values.
# Its built-in _pickle_attrs stores them in .pkl instead, without changing energetics.
for attr in ['adsorbate_coefficient_matrix', 'coefficient_matrix', 'kBT',
             'total_coefficient_dict', 'transition_state_coefficient_matrix',
             'transition_state_scaling_matrix', 'coverage_map', 'rate_map',
             'production_rate_map']:
    if attr not in model._pickle_attrs:
        model._pickle_attrs.append(attr)

# Ignore old map caches which were saved with only ['coverage', 'rate'].
model.force_recalculation = True
model.run()
print('Model directory:', Path.cwd())
print('Output variables:', model.output_variables)


In [ ]:
import math

resolution = model.resolution
if isinstance(resolution, (list, tuple)):
    expected_points = math.prod(int(v) for v in resolution)
else:
    expected_points = int(resolution) ** len(model.descriptor_names)

print('Expected descriptor grid points:', expected_points)
for name in ['production_rate', 'rate', 'coverage']:
    data = getattr(model, name + '_map', None)
    actual = len(data) if data is not None else 0
    print(name, 'points:', actual)
    if actual != expected_points:
        raise RuntimeError(f'{name}: expected {expected_points}, got {actual}')

print('Coverage labels:', model.output_labels['coverage'])
bad = []
for point, coverages in model.coverage_map:
    vals = [float(v) for v in coverages]
    if (not all(math.isfinite(v) for v in vals) or
            min(vals) < -1e-8 or max(vals) > 1+1e-8 or sum(vals) > 1+1e-6):
        bad.append((point, vals))
print('Coverage physical-range warnings:', len(bad))
for item in bad[:5]:
    print('WARNING:', item)


In [ ]:
import csv
from pathlib import Path

with Path('energies.txt').open(encoding='utf-8', newline='') as fh:
    source_rows = list(csv.DictReader(fh, delimiter='\t'))
by_surface = {}
for row in source_rows:
    if row['surface_name'] != 'None':
        by_surface.setdefault(row['surface_name'], {})[row['species_name']] = float(row['formation_energy'])

records = []
for surface in model.surface_names:
    inp = by_surface[surface]
    scaled = model.scaler.get_electronic_energies([inp['R'], inp['P']])
    actual_RP = float(scaled['RP_s'])
    barrier_actual = max(0., actual_RP-inp['R'], inp['P']-inp['R'])
    records.append({
        'surface': surface, 'R_input': inp['R'], 'P_input': inp['P'],
        'RP_input': inp['RP'], 'RP_scaled': actual_RP,
        'RP_scaling_error_eV': actual_RP-inp['RP'],
        'barrier_input_eV': max(0., inp['RP']-inp['R']),
        'barrier_scaled_eV': barrier_actual,
        'R_scaling_error_eV': float(scaled['R_s'])-inp['R'],
        'P_scaling_error_eV': float(scaled['P_s'])-inp['P']})

with Path('ts_scaling_audit.csv').open('w',encoding='utf-8-sig',newline='') as fh:
    writer = csv.DictWriter(fh,fieldnames=list(records[0]))
    writer.writeheader()
    writer.writerows(records)
print('Saved:', Path('ts_scaling_audit.csv').resolve())
for row in sorted(records,key=lambda r:abs(r['RP_scaling_error_eV']),reverse=True)[:8]:
    print(f"{row['surface']}: input Ea={row['barrier_input_eV']:.3f}, "
          f"scaled Ea={row['barrier_scaled_eV']:.3f}, "
          f"TS error={row['RP_scaling_error_eV']:+.3f} eV")
if any(abs(r['RP_scaling_error_eV']) > 0.05 for r in records):
    print('WARNING: RP is changed by linear fitting; plotted TOF is NOT exact per-surface NEB kinetics.')


In [ ]:
# Stable plotting: force every CatMAP mpf/np scalar to plain float BEFORE matplotlib.
# This avoids "ufunc isfinite not supported" from dtype=object.
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.colors import TwoSlopeNorm
from pathlib import Path
from datetime import datetime

run_dir = Path.cwd()
backup_dir = run_dir / "plots" / ("rerun_" + datetime.now().strftime("%Y%m%d_%H%M%S"))
def save_figure(fig, filename):
    target = run_dir / filename
    try:
        fig.savefig(target, bbox_inches="tight")
    except PermissionError:
        backup_dir.mkdir(parents=True, exist_ok=True)
        target = backup_dir / filename
        fig.savefig(target, bbox_inches="tight")
        print("WARNING: existing PDF is locked; saved to", target)
    finally:
        plt.close(fig)
    print("Saved", target)
    return target

def get_float_map(var):
    raw = getattr(model, var + "_map", None)
    if not raw:
        raise RuntimeError(var + "_map missing. Re-run the model first.")
    xy = np.asarray([point for point, values in raw], dtype=np.float64)
    zz = np.asarray([[float(v) for v in values] for point, values in raw],
                    dtype=np.float64)
    if not np.isfinite(xy).all() or not np.isfinite(zz).all():
        raise ValueError(var + " contains NaN/inf. Check convergence.")
    return xy, zz

def draw_map(ax, points, values, title, mode="linear", min_value=None, max_value=None):
    x, y = points.T
    if mode == "log_positive":
        if (values < 0).any():
            raise ValueError(title + ": net production contains negative values; cannot log plot")
        z = np.log10(np.maximum(values, 1e-30))
        levels = 18
        unit = "log10(max(rate, 1e-30))"
    elif mode == "signed":
        # Linear signed net rate: no invalid log10 and no loss of reverse-rate sign.
        z = values
        levels = 18
        unit = "signed net rate"
    else:
        z = values
        levels = 18
        unit = "coverage"
    if np.allclose(z.max(), z.min(), atol=1e-50, rtol=0):
        im = ax.scatter(x, y, c=z, s=40, cmap="viridis")
    else:
        im = ax.tricontourf(x, y, z, levels=levels, cmap="viridis")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Gf(R*) [eV]")
    ax.set_ylabel("Gf(P*) [eV]")
    ax.set_aspect("auto")
    ax.figure.colorbar(im, ax=ax, label=unit)
    return ax

xy, reaction_rate = get_float_map("rate")
rate_labels = ["Step " + str(i+1) for i in range(reaction_rate.shape[1])]
fig, ax = plt.subplots(figsize=(7, 5))
draw_map(ax, xy, reaction_rate[:,1] if reaction_rate.shape[1]>1 else reaction_rate[:,0],
         "Epoxidation intermediate conversion (signed rate)", mode="signed")
save_figure(fig, "rate.pdf")

def save_multipage(plots, filename):
    target = run_dir / filename
    try:
        with PdfPages(target) as pdf:
            for fig in plots:
                pdf.savefig(fig, bbox_inches="tight")
    except PermissionError:
        backup_dir.mkdir(parents=True, exist_ok=True)
        target = backup_dir / filename
        with PdfPages(target) as pdf:
            for fig in plots:
                pdf.savefig(fig, bbox_inches="tight")
        print("WARNING: output PDF locked; used",target)
    finally:
        for fig in plots:
            plt.close(fig)
    print("Saved", target)
save_multipage([
    (lambda i: (lambda f,a: (draw_map(a, xy, reaction_rate[:,i],
                    "Elementary reaction: " + rate_labels[i], mode="signed"), f)[1])
                 (*plt.subplots(figsize=(7,5))))(i)
    for i in range(reaction_rate.shape[1])
], "all_rates.pdf")

xy_p, production = get_float_map("production_rate")
product_labels = list(model.output_labels["production_rate"])
if "C6H12O_g" not in product_labels:
    raise ValueError("C6H12O_g is absent from production-rate labels")
epoxide_rates = production[:, product_labels.index("C6H12O_g")]
mode = "log_positive" if (epoxide_rates >= 0).all() else "signed"
for filename,title in [("production_rate.pdf","Epoxide production rate"),
                       ("pretty_production_rate.pdf","Epoxide activity map"),
                       ("volcano_R_P.pdf","Epoxidation activity map (R*, P* descriptors)")]:
    fig,ax = plt.subplots(figsize=(7.2,5.4))
    draw_map(ax,xy_p,epoxide_rates,title,mode=mode)
    save_figure(fig,filename)

xy_c, coverage = get_float_map("coverage")
coverage_labels = list(model.output_labels["coverage"])
if coverage.shape[1] > len(coverage_labels):
    coverage_labels += ["vacant site"] * (coverage.shape[1]-len(coverage_labels))
for idx,label in enumerate(coverage_labels):
    fig,ax=plt.subplots(figsize=(7,5))
    draw_map(ax,xy_c,coverage[:,idx],str(label)+" coverage",mode="linear")
    save_figure(fig, "coverage_" + str(idx+1) + ".pdf")
fig,ax=plt.subplots(figsize=(7,5))
idx=coverage_labels.index("R_s")
draw_map(ax,xy_c,coverage[:,idx],"R* coverage")
save_figure(fig,"R_P_coverage.pdf")
print("Plotting complete. These volcano-like surfaces use R/P, NOT C/O/H descriptors.")


In [ ]:
# Independent scaling diagnostic, without CatMAP ScalingAnalysis(parameter_dict=None).
# Plots input vs predicted RP (eV) to reveal where linear scaling changes the barrier.
import numpy as np
import matplotlib.pyplot as plt

names = [r["surface"] for r in records]
xx = np.array([r["R_input"] for r in records], dtype=float)
raw_rp = np.array([r["RP_input"] for r in records], dtype=float)
fit_rp = np.array([r["RP_scaled"] for r in records], dtype=float)
fig,ax=plt.subplots(figsize=(8,6))
ax.scatter(xx, raw_rp, label="Input effective RP", marker="o")
ax.scatter(xx, fit_rp, label="CatMAP scaled RP", marker="x")
for i,name in enumerate(names):
    ax.annotate(name, (xx[i], raw_rp[i]), xytext=(3,4),
                textcoords="offset points", fontsize=7)
ax.set_xlabel("Gf(R*) [eV]")
ax.set_ylabel("Gf(RP*) [eV]")
ax.set_title("RP scaling: input vs CatMAP fitted (provisional)")
ax.legend()
ax.grid(alpha=.2)
save_figure(fig,"scaling.pdf")


In [ ]:
import csv
from pathlib import Path

variable = 'production_rate'
labels = [str(v) for v in model.output_labels[variable]]
out = Path(variable+'_table.txt')
with out.open('w',encoding='utf-8',newline='') as fh:
    writer = csv.writer(fh,delimiter='\t')
    writer.writerow(['descriptor-'+d for d in model.descriptor_names] + labels)
    for point, rates in getattr(model,variable+'_map'):
        writer.writerow([float(x) for x in point]+[float(x) for x in rates])
print('Wrote',out.resolve())


In [ ]:
point, cvg = model.coverage_map[0]
print('descriptors:',point)
print('species:',model.output_labels['coverage'])
print('coverages:',[float(v) for v in cvg])


In [ ]:
# Optional expensive sensitivity calculation; run only if explicitly needed.
# for name in ['rate_control', 'selectivity_control', 'rxn_order']:
#     if name not in model.output_variables:
#         model.output_variables.append(name)
# model.run()
# mm = analyze.MatrixMap(model)
# mm.plot_variable = 'rate_control'
# mm.log_scale = False
# mm.min, mm.max = -2, 2
# mm.plot(save='rate_control.pdf')


In [ ]:
# C/O/H descriptor activity diagrams: require real per-catalyst E_C/E_O/E_H.
# These are separate from the R/P microkinetic descriptor grid.
from pathlib import Path
import csv
from runpy import run_path

descriptor_file = Path("descriptor_C_O_H.csv")
with descriptor_file.open(encoding="utf-8-sig", newline="") as fh:
    entries = list(csv.DictReader(fh))
missing = [row.get("surface","?") for row in entries
           if any(not row.get(key,"").strip() for key in ("E_C","E_O","E_H"))]
if missing:
    print("C/O/H volcano plots not generated: no verified E_C/E_O/E_H for "
          + ", ".join(missing))
    print("Complete descriptor_C_O_H.csv from your DFT definitions first.")
else:
    run_path("plot_C_O_H_volcano.py", run_name="__main__")
